# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedahmed02/Flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions


### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages were longer and younger on average than declining pages
(3.2K vs 2.3K words; 184 vs 230 days). The paper appropriately describes this as an
observational comparison rather than a causal result.

**Methodology question:** The growing/declining label is derived from observed trend direction.
It would be useful to clarify that the features being compared are measured strictly before the
outcome window. If a feature contains information from the same window used to define the
trend label, the comparison could partially reflect the outcome itself.

This is a constructive question about temporal alignment, not a claim that the finding is wrong.

### Finding 2 — Logistic Regression: What Predicts Growth?

The exploratory ML appendix reports 71% holdout accuracy for logistic regression predicting
growing versus declining content. The paper correctly describes this result as exploratory
and notes that the ML appendix is secondary to the direct portfolio comparisons.

**Methodology question:** The growth label is derived from trend direction, while the reported
validation uses an 80/20 holdout. It would strengthen the result to clarify whether the holdout
was grouped by client or separated in time. A random split can place content from the same
client in both training and test sets, so part of the measured accuracy could reflect
within-client similarity rather than generalization to unseen clients or future content.

This is why I will re-evaluate my own Week-5 model with a client-grouped split.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic imports and reproducibility

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

print("Environment ready.")

Environment ready.


## 2. My model under an honest split (before/after)

For this audit, I compare a row-level random split with the grouped split by `client_hash_id` used in Week 5.

The random split is useful as a comparison, but it allows content from the same client to appear in both training and test data. This can make evaluation easier because client-specific patterns may be shared across both sets.

The grouped split is stricter for my decision question because all content from a test client is kept separate from the training clients. This better measures whether the model can rank content for previously unseen clients.

Both evaluations use the same March features, April outcome definition, Logistic Regression model, and Precision@50 metric. The comparison is directional evidence about the effect of the validation design, not proof that one score will generalize to every future dataset.

In [3]:
!pip -q install -U duckdb huggingface_hub pyarrow scikit-learn

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42
K = 50

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HTTP,
    EXTRA_HTTP_HEADERS MAP {{
        'Authorization': 'Bearer {HF_TOKEN}'
    }}
);
""")

MARCH_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

APRIL_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

dataset = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_sum_position) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet('{MARCH_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks

    FROM read_parquet('{APRIL_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.*,
    a.april_clicks,

    CASE
        WHEN a.april_clicks > m.gsc_clicks THEN 1
        ELSE 0
    END AS label

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
""").fetchdf()

FEATURES = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

TARGET = "label"
GROUP = "client_hash_id"

X = dataset[FEATURES].copy()
y = dataset[TARGET].astype(int)
groups = dataset[GROUP]

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=SEED
        ))
    ])

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_idx = np.argsort(-scores)[:k]
    return y_true[top_idx].mean()


# ============================================================
# BEFORE: random row-level split
# ============================================================

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y
    )
)

random_model = make_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_p50 = precision_at_k(
    y_test_random,
    random_scores,
    K
)


# ============================================================
# AFTER: grouped split by client
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

grouped_model = make_model()

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_p50 = precision_at_k(
    y_test_grouped,
    grouped_scores,
    K
)


# ============================================================
# Validation checks
# ============================================================

train_clients = set(
    dataset.iloc[train_idx][GROUP]
)

test_clients = set(
    dataset.iloc[test_idx][GROUP]
)

client_overlap = len(
    train_clients & test_clients
)


# ============================================================
# Before / after comparison
# ============================================================

validation_comparison = pd.DataFrame({
    "validation_design": [
        "Random row-level split (before)",
        "Grouped by client (after)"
    ],
    "precision_at_50": [
        random_p50,
        grouped_p50
    ],
    "test_rows": [
        len(X_test_random),
        len(X_test_grouped)
    ]
})

print("Grouped split client overlap:", client_overlap)

display(validation_comparison)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 114.4 MB/s eta 0:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grouped split client overlap: 0


,validation_design,precision_at_50,test_rows
0,Random row-level split (before),0.40,31710
1,Grouped by client (after),0.38,21104


## 3. Leakage audit

I audited the final feature set against the decision time and label definition.

The decision-time features are calculated from March:

- `gsc_clicks`
- `gsc_impressions`
- `gsc_avg_position`
- `ga4_sessions`
- `scroll_events`

The label uses April clicks and is defined as whether April clicks are greater than March clicks.

`april_clicks` is excluded from the model because it is future information and is directly involved in constructing the label. Including it would be direct target leakage.

The remaining model features are measured from March, before the April outcome. Based on their timing, I did not find direct future leakage in the final feature set.

A remaining limitation is that some March features may contain client-specific patterns. This is one reason the grouped-by-client validation is important: it tests whether performance remains useful when the model is evaluated on unseen clients.

This audit does not prove the model is free from every possible source of leakage. It documents the timing and role of each feature and excludes known future information from training.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_audit = pd.DataFrame([
    {
        "feature": "gsc_clicks",
        "source_time": "March",
        "used_in_label": "Yes, as comparison reference",
        "leakage_status": "Allowed with caution",
        "reason": (
            "Available at decision time. It is also used as the March "
            "reference when constructing the future comparison label, "
            "but it does not contain April information."
        )
    },
    {
        "feature": "gsc_impressions",
        "source_time": "March",
        "used_in_label": "No",
        "leakage_status": "No direct future leakage found",
        "reason": (
            "Measured before the April outcome and not used directly "
            "to construct the label."
        )
    },
    {
        "feature": "gsc_avg_position",
        "source_time": "March",
        "used_in_label": "No",
        "leakage_status": "No direct future leakage found",
        "reason": (
            "Calculated from March search data before the April outcome."
        )
    },
    {
        "feature": "ga4_sessions",
        "source_time": "March",
        "used_in_label": "No",
        "leakage_status": "No direct future leakage found",
        "reason": (
            "Measured during March and available before the future outcome."
        )
    },
    {
        "feature": "scroll_events",
        "source_time": "March",
        "used_in_label": "No",
        "leakage_status": "No direct future leakage found",
        "reason": (
            "Measured during March and available before the future outcome."
        )
    },
    {
        "feature": "april_clicks",
        "source_time": "April",
        "used_in_label": "Yes",
        "leakage_status": "Excluded — direct leakage",
        "reason": (
            "Future information and directly used to construct the label."
        )
    }
])

display(leakage_audit)

,feature,source_time,used_in_label,leakage_status,reason
0,gsc_clicks,March,"Yes, as comparison reference",Allowed with caution,Available at decision time. It is also used as...
1,gsc_impressions,March,No,No direct future leakage found,Measured before the April outcome and not used...
2,gsc_avg_position,March,No,No direct future leakage found,Calculated from March search data before the A...
3,ga4_sessions,March,No,No direct future leakage found,Measured during March and available before the...
4,scroll_events,March,No,No direct future leakage found,Measured during March and available before the...
5,april_clicks,April,Yes,Excluded — direct leakage,Future information and directly used to constr...


## 4. Claim rewrite

### Original claim

"The Logistic Regression model is a clear improvement over the baseline and can predict which content will increase in organic clicks."

### Audited rewrite

"In this dataset and evaluation setup, I observed that the Logistic Regression model measured higher Precision@50 than the Week-4 rule baseline on the grouped-by-client test split. The before/after validation comparison also measured how the reported ranking performance changes when client overlap is removed from evaluation.

These results are directional evidence that the model may be useful as decision-support for prioritizing content review. They do not prove that the model will predict future click increases accurately for every client or dataset."

This version is more careful because the measured result depends on the selected months, feature definitions, model, metric, and validation design.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_check = pd.DataFrame({
    "claim_type": [
        "Original",
        "Audited rewrite"
    ],
    "claim": [
        "The Logistic Regression model is a clear improvement over the baseline and can predict which content will increase in organic clicks.",
        (
            "In this dataset and evaluation setup, I observed that the "
            "Logistic Regression model measured higher Precision@50 than "
            "the Week-4 rule baseline on the grouped-by-client test split. "
            "The result is directional evidence for decision-support, not "
            "proof of general future performance."
        )
    ]
})

display(claim_check)

,claim_type,claim
0,Original,The Logistic Regression model is a clear impro...
1,Audited rewrite,"In this dataset and evaluation setup, I observ..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.